# 🤖 Notebook Guide — ML Fundamentals for HackerRank

## Learning Objectives
- [ ] Build sklearn Pipelines that prevent data leakage (most common exam mistake)
- [ ] Perform stratified cross-validation and interpret multiple metrics simultaneously
- [ ] Visualize high-dimensional sequence data with PCA
- [ ] Handle class imbalance correctly (class_weight, SMOTE, appropriate metrics)
- [ ] Tune hyperparameters with GridSearchCV and RandomizedSearchCV
- [ ] Understand feature engineering from DNA sequences

## Why This Notebook Matters for HackerRank
The HackerRank ML Basic + Intermediate certifications test you on:
1. Correctly splitting data (no leakage)
2. Choosing the right metric (accuracy is wrong for imbalanced data)
3. Cross-validation setup
4. Feature scaling inside pipelines

---

## 🤖 Claude Integration — Copy These Prompts

**On data leakage (most critical concept):**
```
Explain data leakage in machine learning with a concrete example.
Why is fitting a StandardScaler on the full dataset before splitting a mistake?
Show me the WRONG way and the RIGHT way using sklearn Pipeline.
What other preprocessing steps can cause leakage (imputation, feature selection)?
```

**On cross-validation:**
```
I'm confused about cross-validation. Explain:
1. Why stratified CV matters for imbalanced classification
2. The difference between cross_val_score and cross_validate
3. How to do nested CV (outer loop for evaluation, inner loop for hyperparameter tuning)
Show code examples with sklearn.
```

**On class imbalance:**
```
My training data has 95% class 0, 5% class 1.
What are all the ways to handle this in sklearn?
Explain: class_weight='balanced', SMOTE, threshold tuning, and choosing F1/MCC over accuracy.
When would I use each approach?
```

**On PCA:**
```
I want to visualize 500 gene expression features in 2D using PCA.
Explain what PCA is doing geometrically.
How do I choose n_components? What does explained_variance_ratio_ tell me?
Should I scale features before PCA? Why?
```

**On GridSearchCV:**
```
Explain GridSearchCV vs RandomizedSearchCV.
When is random search better than grid search?
How do I set up a param_grid for a Pipeline where I want to tune both the
preprocessor and the model?
Show me how to avoid data leakage when using GridSearchCV.
```

---

## Key Patterns to Memorize

```python
# CORRECT pattern — always use Pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

pipe = Pipeline([
    ('scaler', StandardScaler()),    # fit only on train, transform both
    ('clf', RandomForestClassifier())
])
pipe.fit(X_train, y_train)          # scaler.fit_transform(X_train), then clf.fit

# WRONG pattern — causes data leakage
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # ← sees test data!
X_train, X_test = train_test_split(X_scaled, ...)  # too late
```

---

## Interview Tip Bank
> **#1 ML interview gotcha**: "Walk me through your preprocessing pipeline." They want to hear you say *Pipeline* and *stratified split*.

> **Metric selection rule**: accuracy=balanced data, F1=imbalanced binary, macro-F1=imbalanced multiclass, AUC-ROC=ranking quality, MCC=most robust single metric.

> **When asked about overfitting**: mention train/val/test split, learning curves, early stopping, regularization (L1/L2), dropout for NNs.

---

## Self-Check
1. What happens if you use `StandardScaler().fit_transform(X)` before `train_test_split`?
2. Name 3 metrics better than accuracy for imbalanced data.
3. What does `cross_validate` return that `cross_val_score` does not?
4. How does `class_weight='balanced'` change how the model trains?
5. In PCA, what does it mean if the first 2 components explain 85% of variance?


## TL;DR — Plain English

**What this notebook does:** Teaches you how to build reliable machine learning pipelines that don't cheat during evaluation, using scikit-learn's best practices.

**After this notebook you can:**
- Build end-to-end ML pipelines with sklearn that safely chain preprocessing and modeling steps
- Run cross-validation correctly to get honest estimates of model performance
- Use GridSearchCV to automatically find the best hyperparameters for any model
- Engineer new features from raw data and avoid the classic data leakage trap

**Why this matters:**
- HackerRank: Pipeline/CV questions appear in nearly every ML certification track; interviewers test if you know what data leakage is
- Isomorphic Labs: Production ML for protein property prediction relies on bulletproof pipelines — leaky code gives falsely optimistic results that mislead drug discovery

**Time:** ~2 hours | **Difficulty:** ⭐⭐ | **Prerequisites:** 00/01 (Python & NumPy basics)

## Beginner Teaching Frame

**Who should fully work through this notebook:** students with little or no ML background.

**How to study it on a first pass:**
- read every markdown section before touching the code
- run the code in small chunks
- paraphrase each concept in your own words
- write one tiny example from memory after finishing

**Common traps:** memorizing syntax without understanding the data flow, skipping probability, and rushing past train/test split discipline.


## Canonical Resource Upgrade

If you need a second explanation, use these first:
- [CS50P](https://cs50.harvard.edu/python/) for programming fundamentals
- [Harvard Stat 110](https://stat110.hsites.harvard.edu/) for probability intuition
- [scikit-learn MOOC](https://inria.github.io/scikit-learn-mooc/) for correct ML workflow
- [PyTorch Tutorials](https://docs.pytorch.org/tutorials/) once you reach the deep learning notebooks


## Prerequisites & Learning Path

| | |
|--|--|
| Prerequisites | 00/01 Python Core — numpy arrays, functions, classes |
| You Are Here | Module 00/02 — ML Fundamentals |
| Next Steps | 04/01 ML for Omics or 05/01 Deep Learning & Fine-tuning |
| Goal | Build and evaluate scikit-learn pipelines; understand bias-variance, cross-validation, metrics |

### From Scratch? Start Here:
1. [Andrew Ng ML Specialization (free audit)](https://www.coursera.org/specializations/machine-learning-introduction) — 3 courses
2. [Kaggle Intro to ML (free)](https://www.kaggle.com/learn/intro-to-machine-learning) — 7 lessons
3. This notebook

**Time:** 10-15 hours | **Difficulty:** ⭐⭐⭐

### Cross-References
- Builds on: 00/01 Python Core
- Used by: 04/01 ML for Omics, 05/01 Deep Learning, 09/01 Model Diagnostics

## What This Notebook Teaches (Plain English)

**Machine learning** = teaching a computer to make predictions from examples instead of writing explicit rules.

**Example**: Instead of writing "if expression > 500 AND gene_name == 'BRCA1' then cancer", you feed the computer thousands of patient samples and it learns the pattern automatically.

This notebook covers the ML tools used in **bioinformatics job interviews** (HackerRank, Isomorphic Labs, Schrödinger, etc.).

### Quick Glossary — Terms You'll See

| Term | Plain English |
|------|--------------|
| **Model** | The learned pattern (a function that takes input → outputs prediction) |
| **Training set** | Examples the model learns from (never evaluated on) |
| **Test set** | New examples used to measure real performance |
| **Feature** | One measurable property of a sample (e.g. gene expression level) |
| **Label** | The answer we want to predict (e.g. cancer / not cancer) |
| **Accuracy** | % of correct predictions — often misleading (see below) |
| **F1 score** | Better metric for imbalanced data: balances precision and recall |
| **Cross-validation** | Train+test multiple times on different splits → honest performance estimate |
| **Pipeline** | Chain of steps (normalize → select features → train model) applied together |
| **Data leakage** | When test data information accidentally influences training → fake good results |
| **Overfitting** | Model memorizes training data but fails on new data |

> **Most common interview mistake**: scaling data on the full dataset before splitting → data leakage. Always split first, then fit the scaler only on training data.

### What You Need Before Starting
- Python lists, dicts, functions (from Notebook 01)
- NumPy arrays (covered briefly inline)
- No prior ML knowledge required — everything is explained

# ML Fundamentals for Bioinformatics
**HackerRank Prep — Theme 0.2**

Covers: supervised/unsupervised learning, evaluation, pipelines, feature engineering — applied to biological datasets.

---

## 1. The ML Workflow

In [ ]:
# ML Fundamentals: Key Patterns Summary
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

np.random.seed(42)
X = np.random.randn(120, 10)
y = (X[:, 0] + 0.5*X[:, 1] > 0).astype(int)

pipe = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression())])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')
print(f"Logistic Regression ROC-AUC: {scores.mean():.3f} ± {scores.std():.3f}")
print("Pipeline: StandardScaler → LogisticRegression (no data leakage)")
print("Cross-validation: evaluate generalization, not training performance")

## 2. Feature Engineering from Sequences

In [ ]:
# Feature Engineering from Biological Sequences
import numpy as np
from collections import Counter

def sequence_features(seq):
    """Extract hand-crafted features from a DNA sequence."""
    seq = seq.upper()
    n = len(seq)
    bases = Counter(seq)
    # Nucleotide frequencies
    feats = {b: bases.get(b, 0)/n for b in 'ACGT'}
    # GC content
    feats['gc'] = (bases['G'] + bases['C']) / n
    # Dinucleotide CpG
    feats['cpg'] = seq.count('CG') / max(1, n-1)
    # AT run fraction
    feats['at_run'] = sum(1 for i in range(n-1) if seq[i] in 'AT' and seq[i+1] in 'AT') / max(1,n-1)
    # Sequence complexity (unique 2-mers / total possible)
    dimers = {seq[i:i+2] for i in range(n-1)}
    feats['complexity'] = len(dimers) / min(16, n-1)
    return feats

seqs = [
    "ATGCGATCGATCGATCGTAGCTAGCTAGCTAGCT",
    "GCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGC",
    "ATATATATATATATATATATATATATATATATATA",
]
print("Sequence features:")
for s in seqs:
    f = sequence_features(s)
    print(f"  {s[:15]}... GC={f['gc']:.2f} CpG={f['cpg']:.3f} complexity={f['complexity']:.2f}")

## 3. Train/Validation/Test Split & Cross-Validation

In [ ]:
# Train/Validation/Test Split & Cross-Validation
import numpy as np
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

np.random.seed(42)
X = np.random.randn(200, 20)
y = (X[:, 0] + X[:, 1] > 0).astype(int)

# Nested CV to avoid overfitting bias
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

from sklearn.pipeline import Pipeline
pipe = Pipeline([('sc', StandardScaler()), ('svm', SVC(kernel='rbf'))])
results = cross_validate(pipe, X, y, cv=outer_cv,
                          scoring=['accuracy', 'roc_auc'], return_train_score=True)

print("Nested cross-validation results:")
for metric in ['test_accuracy', 'test_roc_auc']:
    scores = results[metric]
    print(f"  {metric}: {scores.mean():.3f} ± {scores.std():.3f}")

# Data leakage demo
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(f"\nTrain size: {len(X_train)}, Test size: {len(X_test)}")
print("Rule: NEVER fit scaler on full dataset before split → data leakage!")

## 4. Pipelines — Proper Way to Scale + Classify

In [ ]:
# ML Fundamentals: Key Patterns Summary
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

np.random.seed(42)
X = np.random.randn(120, 10)
y = (X[:, 0] + 0.5*X[:, 1] > 0).astype(int)

pipe = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression())])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')
print(f"Logistic Regression ROC-AUC: {scores.mean():.3f} ± {scores.std():.3f}")
print("Pipeline: StandardScaler → LogisticRegression (no data leakage)")
print("Cross-validation: evaluate generalization, not training performance")

## 5. Evaluation Metrics — Bioinformatics Context

In [ ]:
# ML Fundamentals: Key Patterns Summary
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

np.random.seed(42)
X = np.random.randn(120, 10)
y = (X[:, 0] + 0.5*X[:, 1] > 0).astype(int)

pipe = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression())])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')
print(f"Logistic Regression ROC-AUC: {scores.mean():.3f} ± {scores.std():.3f}")
print("Pipeline: StandardScaler → LogisticRegression (no data leakage)")
print("Cross-validation: evaluate generalization, not training performance")

## 6. Dimensionality Reduction — PCA on Genomic Features

In [ ]:
# ML Fundamentals: Key Patterns Summary
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

np.random.seed(42)
X = np.random.randn(120, 10)
y = (X[:, 0] + 0.5*X[:, 1] > 0).astype(int)

pipe = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression())])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')
print(f"Logistic Regression ROC-AUC: {scores.mean():.3f} ± {scores.std():.3f}")
print("Pipeline: StandardScaler → LogisticRegression (no data leakage)")
print("Cross-validation: evaluate generalization, not training performance")

## 7. Handling Imbalanced Biological Datasets

In [ ]:
# ML Fundamentals: Key Patterns Summary
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

np.random.seed(42)
X = np.random.randn(120, 10)
y = (X[:, 0] + 0.5*X[:, 1] > 0).astype(int)

pipe = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression())])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')
print(f"Logistic Regression ROC-AUC: {scores.mean():.3f} ± {scores.std():.3f}")
print("Pipeline: StandardScaler → LogisticRegression (no data leakage)")
print("Cross-validation: evaluate generalization, not training performance")

## 8. Hyperparameter Tuning

In [ ]:
# ML Fundamentals: Key Patterns Summary
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

np.random.seed(42)
X = np.random.randn(120, 10)
y = (X[:, 0] + 0.5*X[:, 1] > 0).astype(int)

pipe = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression())])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')
print(f"Logistic Regression ROC-AUC: {scores.mean():.3f} ± {scores.std():.3f}")
print("Pipeline: StandardScaler → LogisticRegression (no data leakage)")
print("Cross-validation: evaluate generalization, not training performance")

## HackerRank ML Assessment Simulation

### Task: Build a cancer vs. normal classifier
Given gene expression data, build the best model with proper evaluation.

In [ ]:
# ML Fundamentals: Key Patterns Summary
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

np.random.seed(42)
X = np.random.randn(120, 10)
y = (X[:, 0] + 0.5*X[:, 1] > 0).astype(int)

pipe = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression())])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')
print(f"Logistic Regression ROC-AUC: {scores.mean():.3f} ± {scores.std():.3f}")
print("Pipeline: StandardScaler → LogisticRegression (no data leakage)")
print("Cross-validation: evaluate generalization, not training performance")

## 📚 Resources

### 1️⃣ Theory — Foundations & Math
- Bias-variance tradeoff — decomposition of generalisation error
- VC dimension and PAC learning theory — why models generalise
- Cross-validation theory: k-fold, stratified, time-series splits
- Bayes theorem applied to classification — Naive Bayes derivation
- Key paper: Domingos (2012) "A Few Useful Things to Know About Machine Learning"

### 2️⃣ Must-Have Popular Resources
#### 🟢 Start Here (Zero ML Background)
- 🆓 **Andrew Ng ML Specialization** — [coursera.org/specializations/machine-learning-introduction](https://www.coursera.org/specializations/machine-learning-introduction) — audit free, 60h, 4M+ students
- 🆓 **Introduction to Statistical Learning (ISLR2)** — [statlearning.com](https://www.statlearning.com/) — free PDF, best applied ML book
- 🆓 **Kaggle Intro to ML** — [kaggle.com/learn/intro-to-machine-learning](https://www.kaggle.com/learn/intro-to-machine-learning) — free, 3h, interactive
- 🆓 **StatQuest YouTube** — [youtube.com/c/joshstarmer](https://www.youtube.com/c/joshstarmer) — every ML concept explained visually
- 📘 Book: *Hands-On Machine Learning* (Aurélien Géron, 3rd ed.) — industry standard
- 🎓 Course: Andrew Ng ML Specialization on Coursera — 4M+ students, free to audit
- ⭐ GitHub: [scikit-learn/scikit-learn](https://github.com/scikit-learn/scikit-learn) — 60k★, read the User Guide
- 🤗 HuggingFace: [AutoTrain](https://huggingface.co/autotrain) — no-code pipeline reference for understanding automation
- 📊 Kaggle: [learn/intro-to-machine-learning](https://www.kaggle.com/learn/intro-to-machine-learning) — free 7-lesson course

### 3️⃣ Practicals — Hands-On
- 💻 Exercise: Titanic end-to-end pipeline — feature engineering, pipeline, cross-validation
- 💻 Exercise: Implement k-fold cross-validation from scratch using only NumPy
- 🔗 GitHub: [ageron/handson-ml3](https://github.com/ageron/handson-ml3) — official Géron notebooks
- 📊 Kaggle: [House Prices](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) — classic regression competition
- 🤗 HuggingFace: [ProteinGym benchmark](https://huggingface.co/datasets/OATML-Markslab/ProteinGym) — protein fitness ML tasks

### 4️⃣ Real-World Problems
- 🏭 Industry: Recursion Pharmaceuticals — genomic image feature pipelines using scikit-learn
- 📊 Dataset: [CAFA-5 Protein Function Prediction](https://www.kaggle.com/competitions/cafa-5-protein-function-prediction) — on Kaggle
- 🔬 Application: Drug response prediction — CCLE cell line data, regression/classification on gene expression features

### 5️⃣ Interview Tips
- ❓ Must know: precision vs recall tradeoff, when to prefer each (disease screening vs spam filter)
- ❓ Must know: data leakage trap — fitting scaler on full dataset before train/test split inflates accuracy
- ⚠️ Common mistake: using accuracy for imbalanced classes (use F1, AUC-ROC, or Matthews CC instead)
- ⚠️ Common mistake: not checking for target leakage in feature set (features derived from the label)
- 💡 Pro tip: always ask "what is the cost of a false positive vs false negative?" before choosing metric

### 6️⃣ Hot Industry Topics
- 🔥 Trending: FLAML AutoML (Microsoft) — [microsoft/FLAML](https://github.com/microsoft/FLAML) automated hyperparameter search
- 🔥 Trending: MLflow for experiment tracking — every serious ML team uses it now
- 🔥 Trending: ZenML / Metaflow for reproducible ML pipelines
- 🚀 Build this: Full scikit-learn pipeline with ColumnTransformer + GridSearchCV on a genomics dataset, tracked with MLflow

### Time Complexity — ML Operations
| Algorithm | Training | Prediction | Space |
|-----------|----------|------------|-------|
| Linear Regression | O(n·d²) | O(d) | O(d) |
| Logistic Regression (SGD) | O(n·d·epochs) | O(d) | O(d) |
| Decision Tree | O(n·d·log n) | O(depth) | O(n) |
| Random Forest (T trees) | O(T·n·d·log n) | O(T·depth) | O(T·n) |
| KNN (brute) | O(1) train | O(n·d) | O(n·d) |
| KNN (KD-tree) | O(n·d·log n) | O(d·log n) | O(n·d) |
| SVM (linear) | O(n·d) | O(d) | O(d) |
| PCA | O(n·d²) | O(d·k) | O(d·k) |
| k-means | O(n·k·iterations) | O(k·d) | O(k·d) |
| Cross-validation (k-fold) | O(k×training) | — | — |

In [ ]:
# Train/Validation/Test Split & Cross-Validation
import numpy as np
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

np.random.seed(42)
X = np.random.randn(200, 20)
y = (X[:, 0] + X[:, 1] > 0).astype(int)

# Nested CV to avoid overfitting bias
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

from sklearn.pipeline import Pipeline
pipe = Pipeline([('sc', StandardScaler()), ('svm', SVC(kernel='rbf'))])
results = cross_validate(pipe, X, y, cv=outer_cv,
                          scoring=['accuracy', 'roc_auc'], return_train_score=True)

print("Nested cross-validation results:")
for metric in ['test_accuracy', 'test_roc_auc']:
    scores = results[metric]
    print(f"  {metric}: {scores.mean():.3f} ± {scores.std():.3f}")

# Data leakage demo
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(f"\nTrain size: {len(X_train)}, Test size: {len(X_test)}")
print("Rule: NEVER fit scaler on full dataset before split → data leakage!")

# 🌍 Real World Problems — ML Fundamentals

---

## Problem 1 — Cancer Type Classification (TCGA PANCAN Dataset)
**Source**: UCI ML Repository — `gene expression cancer RNA-Seq` (PANCAN, 801 samples, 5 tumor types)
**Dataset**: https://archive.ics.uci.edu/dataset/401/gene+expression+cancer+rna+seq
**Skills**: Pipeline, stratified CV, feature selection, imbalanced data

**Task**: Classify tumor type (BRCA, KIRC, COAD, LUAD, PRAD) from RNA-seq features using a proper ML pipeline.

```python
# Load dataset (download from UCI or use this direct loader)
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Download: https://archive.ics.uci.edu/dataset/401/gene+expression+cancer+rna+seq
# Unzip and load
# df_X = pd.read_csv('data.csv', index_col=0)
# df_y = pd.read_csv('labels.csv', index_col=0)

# Simulate the structure (801 samples, 20531 genes, 5 classes) for local testing
np.random.seed(42)
n_samples, n_genes = 801, 20531
classes = ['BRCA', 'KIRC', 'COAD', 'LUAD', 'PRAD']
class_counts = [300, 146, 78, 141, 136]
y_raw = np.concatenate([[c]*n for c, n in zip(classes, class_counts)])
np.random.shuffle(y_raw)
X = np.random.lognormal(3, 1.5, (n_samples, n_genes))
le = LabelEncoder()
y = le.fit_transform(y_raw)

# YOUR TASK: Build a Pipeline and evaluate
# Requirements:
# 1. Train/test split (stratified)
# 2. Pipeline: StandardScaler -> SelectKBest(k=500) -> RandomForest
# 3. 5-fold stratified CV with f1_macro metric
# 4. Print top 10 most important genes
# 5. PCA visualization (2D) colored by tumor type
```

**Key learning**: This is the EXACT type of ML assessment used in bioinformatics company interviews.
**Expected accuracy**: ~95%+ with proper feature selection.

---

## Problem 2 — Sepsis Early Warning (Clinical Vital Signs)
**Source**: MIMIC-III (MIT clinical database) — simplified version
**Skills**: Pipeline, leakage prevention, temporal cross-validation

**Task**: Predict sepsis onset from ICU vital signs. The critical challenge: you CANNOT use future data (temporal split, not random split).

```python
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# Simulate ICU time series (each row = 1 hour for one patient)
np.random.seed(0)
n = 2000
vitals = pd.DataFrame({
    'heart_rate':      np.random.normal(80, 15, n),
    'blood_pressure':  np.random.normal(120, 20, n),
    'temperature':     np.random.normal(37, 0.8, n),
    'respiratory_rate': np.random.normal(16, 4, n),
    'oxygen_sat':      np.random.normal(97, 2, n),
    'hour':            np.arange(n),
})
# Add sepsis signal (patients who become septic have elevated HR + low BP)
sepsis_onset = ((vitals['heart_rate'] > 100) & (vitals['blood_pressure'] < 90)).astype(int)
vitals['sepsis'] = (sepsis_onset + np.random.binomial(1, 0.03, n)).clip(0, 1)

X = vitals.drop(columns=['sepsis', 'hour'])
y = vitals['sepsis']

# WRONG: random split leaks future → artificially high performance
# CORRECT: time-based split

# YOUR TASK:
# 1. Use TimeSeriesSplit(n_splits=5) for CV
# 2. Build a Pipeline (StandardScaler + GBM)
# 3. Report AUC-ROC and Average Precision (better for imbalanced)
# 4. Compare: what AUC do you get with random split vs time split?
tscv = TimeSeriesSplit(n_splits=5)
pipe = Pipeline([('sc', StandardScaler()), ('clf', GradientBoostingClassifier())])
# Complete the evaluation loop...
```

**Real impact**: This class of model is used in hospitals for early sepsis alerting.

---

## Problem 3 — Drug Response Prediction (Pharmacogenomics)
**Source**: GDSC (Genomics of Drug Sensitivity in Cancer) dataset
**Skills**: MultiOutputRegressor, feature engineering, cross-validation

**Task**: Given cancer cell line gene expression profiles, predict IC50 response to 5 drugs simultaneously.

```python
# This mirrors the GDSC dataset structure
# Real data: https://www.cancerrxgene.org/downloads/bulk_download
np.random.seed(42)
n_cell_lines = 500
n_genes = 200
n_drugs = 5

X_expr = np.random.lognormal(3, 1, (n_cell_lines, n_genes))  # gene expression
y_ic50 = np.random.lognormal(2, 1.5, (n_cell_lines, n_drugs))  # log IC50 values

drug_names = ['Erlotinib', 'Lapatinib', 'Nutlin-3a', 'Paclitaxel', 'Sorafenib']

# YOUR TASK:
# 1. MultiOutputRegressor wrapping Ridge regression
# 2. Per-drug R² and Spearman correlation
# 3. Feature importance: which genes predict drug sensitivity best?
# 4. Bonus: compare MultiOutputRegressor vs training one model per drug
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import Ridge

model = MultiOutputRegressor(Ridge(alpha=1.0))
# Complete the full pipeline here...
```

---

## Datasets to Practice On
| Dataset | Source | Task |
|---------|--------|------|
| PANCAN gene expression | UCI ML Repo | Cancer type classification |
| GDSC drug response | cancerrxgene.org | Drug sensitivity regression |
| Breast Cancer Wisconsin | sklearn built-in | Binary classification |
| MIMIC-III (ICU) | physionet.org | Clinical time-series prediction |


---

## Resources
| Resource | URL | Purpose |
|----------|-----|---------|
| TCGA PANCAN Dataset | kaggle.com/datasets/crawford/gene-expression-cancer-rna-seq | 5-cancer RNA-seq classification |
| MIMIC-III Clinical Data | kaggle.com/datasets/drscarlat/mimic3-patient-ich | ICU prediction tasks |
| GDSC Drug Response | github.com/CancerRxGene/gdsctools | Drug sensitivity ML |
| scikit-learn Pipelines | github.com/scikit-learn/scikit-learn | ML pipeline reference |
| CAFA5 Protein Function | kaggle.com/competitions/cafa-5-protein-function-prediction | $57K bioinformatics competition |
| Cancer Gene Expression | huggingface.co/datasets/mstz/pancan | PANCAN dataset on HuggingFace |

## Mastery Check

Before moving on, make sure you can:
1. explain the core concept of this notebook in plain English without looking
2. write a small toy example from scratch
3. identify one common mistake and explain why it is wrong
4. say whether you should revisit math, Python, or ML basics before continuing


---
## 🎯 Predict Before Running — ML Intuition Check

Before running the model training cells, answer:

1. You train a RandomForestClassifier on 1000 samples with 50 features. You get 98% training accuracy and 61% test accuracy. Name the phenomenon and at least two fixes.
2. You have a dataset where 95% of samples are class 0, 5% are class 1. A model predicts class 0 for everything. What accuracy does it achieve? Why is accuracy the wrong metric here?
3. If you double the regularization parameter `C` in an SVM, does the decision boundary become more or less complex?

In [ ]:
# PREDICTION VERIFICATION

import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

np.random.seed(42)

# 1. Overfitting demonstration
X, y = make_classification(n_samples=1000, n_features=50, n_informative=5,
                            n_redundant=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rf = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42)  # unlimited depth
rf.fit(X_train, y_train)
print("1. OVERFITTING DEMO:")
print(f"   Train accuracy: {accuracy_score(y_train, rf.predict(X_train)):.3f}")
print(f"   Test  accuracy: {accuracy_score(y_test, rf.predict(X_test)):.3f}")
print("   → Large train-test gap = overfitting")

# 2. Imbalanced dataset — accuracy paradox
X_imb = np.random.randn(1000, 5)
y_imb = np.zeros(1000, dtype=int)
y_imb[:50] = 1  # 5% positive class

always_zero = np.zeros(1000, dtype=int)
print(f"\n2. IMBALANCED DATASET ACCURACY PARADOX:")
print(f"   'Predict all zeros' accuracy: {accuracy_score(y_imb, always_zero):.3f}")
print(f"   'Predict all zeros' F1 score: {f1_score(y_imb, always_zero, zero_division=0):.3f}")
print("   → 95% accuracy is misleading! Use F1, precision-recall, or ROC-AUC instead.")

# 3. SVM C parameter effect
from sklearn.svm import SVC
from sklearn.inspection import DecisionBoundaryDisplay

X_2d, y_2d = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                  n_clusters_per_class=1, random_state=42)
print(f"\n3. SVM C PARAMETER EFFECT:")
for C in [0.01, 1.0, 100.0]:
    svm = SVC(C=C, kernel='rbf').fit(X_2d, y_2d)
    print(f"   C={C:6.2f}: Train={svm.score(X_2d, y_2d):.3f} — "
          f"{'complex boundary (risk overfit)' if C >= 10 else 'smooth boundary (regularized)'}")
print("   → Higher C = less regularization = more complex boundary = risk of overfitting")

---
## 🐛 Debug This — Wrong Evaluation Metrics

The following code evaluates a classifier but contains a critical methodological error that would invalidate a paper or interview answer. Find it.

In [ ]:
# DEBUG EXERCISE — Data Leakage in Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
import numpy as np

np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=20, random_state=42)

# ─── BUGGY PIPELINE (data leakage) ───
scaler_buggy = StandardScaler()
X_scaled = scaler_buggy.fit_transform(X)   # BUG: scales on ALL data before split

X_train_b, X_test_b = X_scaled[:400], X_scaled[400:]
y_train, y_test = y[:400], y[400:]

model_buggy = LogisticRegression().fit(X_train_b, y_train)
buggy_score = model_buggy.score(X_test_b, y_test)
print(f"Buggy pipeline score:  {buggy_score:.4f}")

# ─── CORRECT PIPELINE (no leakage) ───
X_train_raw, X_test_raw = X[:400], X[400:]
scaler_correct = StandardScaler()
scaler_correct.fit(X_train_raw)         # CORRECT: fit on train ONLY
X_train_c = scaler_correct.transform(X_train_raw)
X_test_c  = scaler_correct.transform(X_test_raw)

model_correct = LogisticRegression().fit(X_train_c, y_train)
correct_score = model_correct.score(X_test_c, y_test)
print(f"Correct pipeline score: {correct_score:.4f}")

print(f"\nDifference: {abs(buggy_score - correct_score):.4f}")
print("\nEXPLANATION:")
print("In the buggy version, StandardScaler sees test data statistics (mean/std)")
print("during fit_transform. This leaks test information into preprocessing.")
print("In production, test data won't be available at training time — the model")
print("will fail in ways that your evaluation didn't catch.")
print("\nFIX: Always use sklearn Pipeline or manually fit scaler on train only.")

# The sklearn way (best practice)
from sklearn.pipeline import Pipeline
pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression())])
pipe.fit(X_train_raw, y_train)
print(f"\nPipeline score (no leakage possible): {pipe.score(X_test_raw, y_test):.4f}")

---
## 📊 Real Data — UCI Breast Cancer Dataset

Let's apply everything to a real medical dataset: the Wisconsin Breast Cancer dataset.
Goal: predict malignant vs. benign from 30 numerical features.
This dataset is hosted at UCI ML Repository and bundled in sklearn.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import numpy as np
import matplotlib.pyplot as plt

# Load real dataset
data = load_breast_cancer()
X, y = data.data, data.target
print(f"Wisconsin Breast Cancer: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Classes: {data.target_names} (0=malignant, 1=benign)")
print(f"Class distribution: {dict(zip(data.target_names, np.bincount(y)))}")

# Proper pipeline with stratified CV
from sklearn.pipeline import Pipeline
models = {
    'Logistic Regression': Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))]),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n5-fold stratified CV results:")
print(f"{'Model':<25} {'Accuracy':>10} {'AUC-ROC':>10}")
print("-" * 48)
for name, model in models.items():
    acc = cross_val_score(model, X, y, cv=cv, scoring='accuracy').mean()
    auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc').mean()
    print(f"{name:<25} {acc:>10.4f} {auc:>10.4f}")

# Best model detailed report
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr)
print("\nRandom Forest detailed report:")
print(classification_report(y_te, rf.predict(X_te), target_names=data.target_names))

# Feature importance
importances = rf.feature_importances_
top5 = np.argsort(importances)[-5:][::-1]
print("Top 5 most predictive features:")
for i in top5:
    print(f"  {data.feature_names[i]}: {importances[i]:.4f}")

print("\n→ This is real medical ML. Notice how feature importances match clinical knowledge:")
print("  'worst radius' and 'worst concave points' are key malignancy indicators.")